In [1]:
%pip install -U gradio requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import base64
import requests
import gradio as gr
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = (
    os.getenv("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_KEY")
)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Vision-capable model for image analysis and health chat
MODEL = "google/gemini-2.5-flash"


def image_to_data_url(image_path):
    """Convert uploaded local image to base64 data URL."""
    if not image_path:
        return None

    extension = os.path.splitext(image_path)[1].lower()

    mime_type = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
        ".gif": "image/gif"
    }.get(extension, "image/jpeg")

    with open(image_path, "rb") as image_file:
        encoded_image = base64.b64encode(
            image_file.read()
        ).decode("utf-8")

    return f"data:{mime_type};base64,{encoded_image}"


def call_ai_doctor(user_message, image_path=None, model=MODEL):
    if not OPENROUTER_API_KEY:
        return "Error: OpenRouter API key was not found in your .env file."

    system_prompt = """
You are a careful AI health-support assistant.

Your responsibilities:
- Provide general health education and supportive guidance.
- Ask useful follow-up questions.
- Explain possible causes without claiming certainty.
- Suggest safe, low-risk next steps.
- Give general home-care guidance only when appropriate.
- Clearly explain when the person should see a doctor.
- Clearly identify emergency warning signs.
- Be calm, respectful, empathetic, and easy to understand.

Medical safety rules:
- You are not a licensed doctor.
- Do not provide a definite diagnosis.
- Do not claim that an image proves a disease.
- Do not prescribe prescription medication.
- Do not recommend dangerous substances, unverified treatments,
  or risky home remedies.
- Do not tell someone to delay emergency care.
- If there is chest pain, difficulty breathing, severe bleeding,
  stroke symptoms, unconsciousness, seizure, severe allergic reaction,
  poisoning, or a life-threatening situation, advise emergency care
  immediately.
- For image analysis, describe only what may be visible and explain
  that a medical professional must confirm the cause.
- Protect privacy and advise users not to upload sensitive identity
  documents.
- Use clear headings where helpful:
  Possible explanation, What you can do now, When to seek care,
  and Questions to consider.
"""

    content = [
        {
            "type": "text",
            "text": user_message
        }
    ]

    if image_path:
        image_data_url = image_to_data_url(image_path)

        content.append({
            "type": "image_url",
            "image_url": {
                "url": image_data_url
            }
        })

    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": content
            }
        ],
        "temperature": 0.2,
        "max_tokens": 800
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "",
        "X-Title": "AI Doctor Health Support"
    }

    try:
        response = requests.post(
            OPENROUTER_URL,
            headers=headers,
            json=payload,
            timeout=60
        )

        response.raise_for_status()

        data = response.json()

        return data["choices"][0]["message"]["content"].strip()

    except Exception as error:
        return f"""
The AI Doctor could not connect to the model.

Technical issue:
{error}

Please check your internet connection, OpenRouter API key,
and selected model. If this is a medical emergency, contact
emergency services or visit the nearest hospital immediately.
"""


def analyze_image(image_path, user_question, model):
    if not image_path:
        return "Please upload an image first."

    question = user_question.strip()

    if not question:
        question = """
Analyze this image from a general health-support perspective.
Describe only visible features, explain possible non-definitive
causes, suggest safe next steps, and state when professional
medical assessment is needed.
"""

    return call_ai_doctor(
        user_message=question,
        image_path=image_path,
        model=model
    )


def home_remedy(image_path, symptoms, model):
    if not symptoms.strip() and not image_path:
        return "Please describe your symptoms or upload an image."

    message = f"""
The user wants safe home-care guidance.

Symptoms or description:
{symptoms}

Provide only low-risk general self-care guidance.
Do not claim a diagnosis.
Do not recommend prescription medication.
Explain when the user should stop home care and seek medical help.
"""

    return call_ai_doctor(
        user_message=message,
        image_path=image_path,
        model=model
    )


def health_chat(message, history, model):
    if not message.strip():
        return "Please enter your health question."

    return call_ai_doctor(
        user_message=message,
        model=model
    )


with gr.Blocks(
    title="AI Doctor Health Support",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # AI Doctor Health Support

        Chat with an AI health-support assistant, upload an image
        for general analysis, and receive safe home-care guidance.

        **Important:** This tool provides general information only.
        It does not replace a qualified healthcare professional.
        For emergencies, contact emergency services or visit a hospital.
        """
    )

    model_selector = gr.Dropdown(
        label="Choose AI model",
        choices=[
            "google/gemini-2.5-flash",
            "openai/gpt-4o-mini"
        ],
        value="google/gemini-2.5-flash"
    )

    with gr.Tab("Chat and Consultation"):

        chat_input = gr.Textbox(
            label="Health question",
            placeholder="Describe your symptoms or ask a health question...",
            lines=6
        )

        chat_button = gr.Button(
            "Consult AI Doctor",
            variant="primary"
        )

        chat_output = gr.Textbox(
            label="AI Doctor Response",
            lines=15,
            interactive=True
        )

        chat_button.click(
            fn=health_chat,
            inputs=[
                chat_input,
                gr.State([]),
                model_selector
            ],
            outputs=chat_output
        )

    with gr.Tab("Upload Image and Analyze"):

        image_input = gr.Image(
            label="Upload image",
            type="filepath"
        )

        image_question = gr.Textbox(
            label="What should be analyzed?",
            placeholder="Example: What could this skin irritation be?",
            lines=4
        )

        analyze_button = gr.Button(
            "Analyze Image",
            variant="primary"
        )

        image_output = gr.Textbox(
            label="Image Analysis",
            lines=18,
            interactive=True
        )

        analyze_button.click(
            fn=analyze_image,
            inputs=[
                image_input,
                image_question,
                model_selector
            ],
            outputs=image_output
        )

    with gr.Tab("Home Remedy Guidance"):

        remedy_image = gr.Image(
            label="Optional symptom image",
            type="filepath"
        )

        symptoms_input = gr.Textbox(
            label="Describe your symptoms",
            placeholder="Describe what you are experiencing...",
            lines=6
        )

        remedy_button = gr.Button(
            "Get Safe Home-Care Guidance",
            variant="primary"
        )

        remedy_output = gr.Textbox(
            label="Home-Care Guidance",
            lines=18,
            interactive=True
        )

        remedy_button.click(
            fn=home_remedy,
            inputs=[
                remedy_image,
                symptoms_input,
                model_selector
            ],
            outputs=remedy_output
        )


demo.launch()

c:\Users\ELEAZAR GIDEON\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ELEAZAR GIDEON\AppData\Local\Temp\ipykernel_9964\2136544359.py:202: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
